In [2]:
from datasets import load_dataset
from jaxtyping import Float
import torch
from transformers.modeling_outputs import ModelOutput
from dataclasses import dataclass

/home/mnikiema/OpenSource/ml-playground/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
data = load_dataset("PhilipMay/stsb_multi_mt", "fr")

In [4]:
train_data = data["train"]
train_data[0]

{'sentence1': 'Un avion est en train de décoller.',
 'sentence2': 'Un avion est en train de décoller.',
 'similarity_score': 5.0}

In [25]:
from transformers import AutoTokenizer, AutoModel

model = AutoModel.from_pretrained("almanach/moderncamembert-base")
tokenizer = AutoTokenizer.from_pretrained("almanach/moderncamembert-base")


How many parameters does the model have?

In [29]:
params = sum(p.numel() for p in model.parameters())
print(f"Number of parameters: {params}")

Number of parameters: 135497472


Encode one of the sentences using the model and print the resulting embedding.

In [7]:
sentence = train_data[0]["sentence1"]
inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs)

In [8]:
last_hidden_state = outputs.last_hidden_state
print(last_hidden_state.shape)

torch.Size([1, 11, 768])


In [ ]:
def mean_pool(model_output, attention_mask: Float[torch.Tensor, "batch seq_len hidden"]) -> Float[torch.Tensor, "batch hidden"]:
    token_embeddings = model_output.last_hidden_state  # [batch, seq_len, hidden]
    mask = attention_mask.unsqueeze(-1).float()        # [batch, seq_len, 1]
    return (token_embeddings * mask).sum(1) / mask.sum(1)  # [batch, hidden]

In [31]:
mask = inputs["attention_mask"]
sentence_embedding = mean_pool(outputs, mask)
print(sentence_embedding.shape)

torch.Size([1, 768])


Cosine Similarity loss:
- Compute the similarity of the sentence 1 and the sentence 2.
- Compare the similarity to the label (between 0 and 1)

In [32]:
normalized_embedding = torch.nn.functional.normalize(sentence_embedding, p=2, dim=1)

In [33]:
embedding1 = normalized_embedding

In [13]:
sentence = train_data[0]["sentence2"]
inputs = tokenizer(sentence, return_tensors="pt")
outputs = model(**inputs)
mask = inputs["attention_mask"]
sentence_embedding = mean_pool(outputs, mask)
normalized_embedding = torch.nn.functional.normalize(sentence_embedding, p=2, dim=1)
embedding2 = normalized_embedding

In [14]:
cossim = torch.nn.functional.cosine_similarity(embedding1, embedding2)
print(cossim)

tensor([1.0000], grad_fn=<SumBackward1>)


In [16]:
true_cossim = train_data[0]["similarity_score"] / 5.0
print(true_cossim)

1.0


In [17]:
loss = torch.nn.functional.mse_loss(cossim, torch.tensor(true_cossim))
print(loss)

tensor(1.4211e-14, grad_fn=<MseLossBackward0>)


/tmp/ipykernel_875000/2597520559.py:1: UserWarning: Using a target size (torch.Size([])) that is different to the input size (torch.Size([1])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  loss = torch.nn.functional.mse_loss(cossim, torch.tensor(true_cossim))


In [18]:
class CosineSimilarityLoss(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, embedding1, embedding2, true_cossim):
        cossim = torch.nn.functional.cosine_similarity(embedding1, embedding2)
        loss = torch.nn.functional.mse_loss(cossim, torch.tensor(true_cossim))
        return loss

In [31]:
class ContrastiveLoss(torch.nn.Module):
    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, embedding1, embedding2, label):
        # label: 1 = similar pair, 0 = dissimilar pair
        dist = 1 - torch.nn.functional.cosine_similarity(embedding1, embedding2)
        loss = label * dist**2 + (1 - label) * torch.clamp(self.margin - dist, min=0)**2
        return loss.mean()

In [19]:
@dataclass
class SimilarityOutput(ModelOutput):
    loss: torch.FloatTensor | None = None

In [39]:
class SModernCamembert(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained("almanach/moderncamembert-base")

    def mean_pool(self, model_output, attention_mask: Float[torch.Tensor, "batch seq_len"]) -> Float[torch.Tensor, "batch hidden"]:
        token_embeddings = model_output.last_hidden_state  # [batch, seq_len, hidden]
        mask = attention_mask.unsqueeze(-1).float()        # [batch, seq_len, 1]
        embeddings = (token_embeddings * mask).sum(1) / mask.sum(1)
        return torch.nn.functional.normalize(embeddings, p=2, dim=-1)

    def forward(self, input_ids1, attention_mask1: Float[torch.Tensor, "batch seq_len hidden_dim"], input_ids2, attention_mask2: Float[torch.Tensor, "batch seq_len hidden_dim"], labels=None):
        embedding1 = self.mean_pool(self.model(input_ids=input_ids1, attention_mask=attention_mask1),attention_mask1)
        embedding2 = self.mean_pool(self.model(input_ids=input_ids2, attention_mask=attention_mask2),attention_mask2)
        loss = None
        if labels is not None:
            cossim = torch.nn.functional.cosine_similarity(embedding1, embedding2)
            loss = torch.nn.functional.mse_loss(cossim, labels)
        return SimilarityOutput(loss=loss)

In [40]:
def preprocess(batch):
    enc1 = tokenizer(batch["sentence1"], truncation=True, max_length=128)
    enc2 = tokenizer(batch["sentence2"], truncation=True, max_length=128)
    return {
        "input_ids1": enc1["input_ids"],
        "attention_mask1": enc1["attention_mask"],
        "input_ids2": enc2["input_ids"],
        "attention_mask2": enc2["attention_mask"],
        "labels": [s / 5.0 for s in batch["similarity_score"]],
    }


In [41]:
from transformers import DataCollatorWithPadding

class PairCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):
        def pad(key_ids, key_mask):
            return self.tokenizer.pad(
                [{"input_ids": f[key_ids], "attention_mask": f[key_mask]} for f in features],
                return_tensors="pt",
            )
        batch1 = pad("input_ids1", "attention_mask1")
        batch2 = pad("input_ids2", "attention_mask2")
        labels = torch.stack([torch.as_tensor(f["labels"]) for f in features]).float()
        return {
            "input_ids1": batch1["input_ids"],
            "attention_mask1": batch1["attention_mask"],
            "input_ids2": batch2["input_ids"],
            "attention_mask2": batch2["attention_mask"],
            "labels": labels,
        }


In [42]:
train_dataset = train_data.map(preprocess, batched=True, remove_columns=train_data.column_names)

In [45]:
from transformers import Trainer, TrainingArguments

model = SModernCamembert()

args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=2,
    num_train_epochs=3,
    learning_rate=2e-5,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=PairCollator(tokenizer),
)

trainer.train()


Step,Training Loss
500,0.068100
1000,0.047400
1500,0.040800
2000,0.038500
2500,0.031100
3000,0.029200
3500,0.018200
4000,0.019300
4500,0.017300
5000,0.016800


TrainOutput(global_step=8625, training_loss=0.022683364253113236, metrics={'train_runtime': 2273.8998, 'train_samples_per_second': 7.585, 'train_steps_per_second': 3.793, 'total_flos': 0.0, 'train_loss': 0.022683364253113236, 'epoch': 3.0})